# Évaluation des prédictions avec ragas (test local)

Remplacer `PREDICTIONS_URI` par l'URI GCS de l'artifact `predictions` produit par l'étape d'inférence.

In [ ]:
import sys

import pandas as pd
from ragas import SingleTurnSample
from ragas.metrics import BleuScore, RougeScore
from ragas.metrics.base import SingleTurnMetric

sys.path.append("..")  # pour importer src.taxonomy depuis notebooks/

from src.taxonomy import SECTIONS, parse_sections

PREDICTIONS_URI = "../data/predictions_sustained.csv"  # ou "gs://<bucket>/pipeline_root/.../predictions"

predictions_df = pd.read_csv(PREDICTIONS_URI, keep_default_na=False)
predictions_df.head()

## 1. Définition des métriques (sans LLM)

In [ ]:
metric_definitions = [RougeScore(), BleuScore()]

In [ ]:
def compute_metrics(
    user_input: str,
    response: str,
    reference: str,
    metric_definitions: list[SingleTurnMetric],
) -> dict[str, float | int]:
    """Compute metrics for a single prediction."""
    sample = SingleTurnSample(user_input=user_input, response=response, reference=reference)
    return {metric.name: metric.single_turn_score(sample) for metric in metric_definitions}


def aggregate_metrics(per_sample_metrics: pd.DataFrame) -> dict[str, float]:
    """Aggregate metrics from per-sample metrics DataFrame."""
    return {f"{name}_mean": float(per_sample_metrics[name].mean()) for name in per_sample_metrics}

## 2. Métriques par exemple

In [ ]:
per_sample_metrics = pd.DataFrame(
    [
        compute_metrics(row.user_input, row.extracted_response, row.reference, metric_definitions)
        for row in predictions_df.itertuples()
    ]
)
pd.concat([predictions_df, per_sample_metrics], axis=1)

## 3. Respect du format pédagogique

Une réponse est conforme si elle contient les quatre parties (Definition, Example, Action, Check), non vides et dans l'ordre. Le composant de la pipeline calcule aussi le niveau de lecture et la présence du profil dans l'exemple.

In [ ]:
def format_adherence(response: str) -> int:
    """1 if the four sections are present, not empty and in order."""
    found = parse_sections(response)
    return int(list(found) == SECTIONS and all(found.values()))


format_scores = predictions_df["extracted_response"].map(format_adherence)
print(f"Réponses conformes : {format_scores.mean():.0%}")
predictions_df.assign(format_adherence=format_scores)

## 4. Métriques agrégées

In [ ]:
aggregate_metrics(per_sample_metrics)